In [ ]:
import akshare as ak
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ═══════════ 配置 ═══════════
INDEX_SYMBOL = "sh000852"       # 中证1000，sh000852；沪深300用sh000300
FUTURES_PREFIX = "IM"            # IM/IF/IC/IH
RISK_FREE_RATE = 0.014
VALUATION_DATE = "2026-08-24"
WINDOW_DAYS = 488
SIGMA_MULTIPLIER = 1.5

# ═══════════ 辅助函数 ═══════════

def get_third_friday(year, month):
    """中金所：合约月份第三个周五"""
    first_day = datetime(year, month, 1)
    days_to_friday = (4 - first_day.weekday()) % 7
    return first_day + timedelta(days=days_to_friday) + timedelta(weeks=2)


def get_window_contracts(prefix, valuation_date):
    """生成488交易日窗口内需要的季月合约
    488交易日 ≈ 730自然日，往前多拉一年确保覆盖早期合约"""
    val = valuation_date if isinstance(valuation_date, datetime) else datetime.strptime(valuation_date, "%Y-%m-%d")
    contracts = []
    for y in range(val.year - 3, val.year + 2):
        for m in [3, 6, 9, 12]:
            expiry = get_third_friday(y, m)
            days_from_val = (expiry - val).days
            # 往前覆盖~2.5年自然日（确保488交易日窗口内的旧合约不遗漏）
            if -910 < days_from_val < 730:
                code = f"{prefix}{y % 100:02d}{m:02d}"
                contracts.append(code)
    return sorted(set(contracts))


# ═══════════ 1. 用 akshare 读现货指数数据 ═══════════
print("=" * 60)
print(f"  读取指数数据: {INDEX_SYMBOL}")
# stock_zh_index_daily 返回 date, open, high, low, close, volume, ...
index_df = ak.stock_zh_index_daily(symbol=INDEX_SYMBOL)
index_df.rename(columns={"date": "date", "close": "close"}, inplace=True)
index_df["date"] = pd.to_datetime(index_df["date"])
index_df = index_df.sort_values("date").reset_index(drop=True)
# 只保留有期货数据的时段（2022-07 之后）
index_df = index_df[index_df["date"] >= "2022-01-01"].reset_index(drop=True)
print(f"  {len(index_df)} 行，{index_df['date'].iloc[0].date()} ~ {index_df['date'].iloc[-1].date()}")

# ═══════════ 2. 用 akshare 读期货数据 ═══════════
print(f"\n  读取期货合约: {FUTURES_PREFIX}")

# 只生成488天窗口内需要的合约
contracts_list = get_window_contracts(FUTURES_PREFIX, VALUATION_DATE)

all_futures = []
for code in contracts_list:
    try:
        df = ak.futures_zh_daily_sina(symbol=code)
        if df is None or len(df) == 0:
            continue
        df.rename(columns={"date": "date", "close": "futures_close"}, inplace=True)
        df["date"] = pd.to_datetime(df["date"])
        df["contract"] = code
        # 从合约代码推导到期日（第三个周五）
        y, m = 2000 + int(code[-4:-2]), int(code[-2:])
        df["expiry"] = get_third_friday(y, m)
        all_futures.append(df)
        print(f"    {code}: {len(df)} 行，到期 {df['expiry'].iloc[0].date()}")
    except Exception as e:
        # 已退市合约可能获取不到，跳过
        pass

if not all_futures:
    print("  ⚠ 没有获取到任何期货合约数据，请检查网络或 akshare 版本")
    exit()

futures_df = pd.concat(all_futures, ignore_index=True)
futures_df = futures_df.sort_values(["date", "contract"]).reset_index(drop=True)
print(f"  合计 {len(futures_df)} 行")